# 🏎️ JetBot 期末專案多功能控制面板 (分測 + 合併最終版)

本筆記本將整合程式分為三個部分，供您在實車落地時進行分步測試與最終合併演示：
- **單元一：純道路跟隨測試 (測試 PID 與轉向)** — 僅載入 ResNet 模型，不佔用 YOLO 與 PyCUDA 資源，適合調校 P/D Gain。
- **單元二：純交通路牌辨識測試 (測試 YOLO 與動作)** — 僅載入 YOLO 模型，測試路標停/走動作是否正常。
- **單元三：合併最終版 (道路跟隨 + 路牌辨識)** — 雙模型並行推論，進行期末考完整賽道演示。

> ⚠️ **重要守則**：每當您要從一個單元切換到另一個單元時，**必須執行該單元最下方的「安全停機與釋放相機」單元格**，否則會出現相機被佔用 (Resource busy) 的錯誤！

---

In [ ]:
## 🛠️ 零、道路跟隨 TensorRT 模型優化轉換 (選用)
# 0-1. 執行 TensorRT 編譯與轉換
import os
import torch
import torchvision.models as models

PYTORCH_MODEL = 'road_following_model/best_steering_model_xy.pth'
TRT_MODEL = 'road_following_model/best_steering_model_xy_trt.pth'
ONNX_MODEL = 'road_following_model/best_steering_model_xy.onnx'
ENGINE_MODEL = 'road_following_model/best_steering_model_xy.engine'

if not os.path.exists(PYTORCH_MODEL):
    print(f"❌ 錯誤：找不到 PyTorch 原始權重檔案：{PYTORCH_MODEL}")
    print("請確認您已將訓練好的 .pth 檔案上傳至對應目錄。")
else:
    print(f"正在載入 PyTorch 權重：{PYTORCH_MODEL}...")
    # 建立 ResNet-18 架構
    model = models.resnet18(pretrained=False)
    model.fc = torch.nn.Linear(512, 2)
    model.load_state_dict(torch.load(PYTORCH_MODEL))
    
    # 搬移至 GPU 並設定為評估模式
    model = model.cuda().eval()
    
    # 1. 測試 PyTorch 運作是否正常
    print("正在進行 PyTorch 模型測試推論...")
    try:
        test_in = torch.zeros((1, 3, 224, 224)).cuda()
        test_out = model(test_in)
        print(f"✓ PyTorch 測試推論成功！輸出值：{test_out.cpu().detach().numpy()}")
    except Exception as e:
        print(f"❌ 錯誤：PyTorch 測試推論失敗！原因：{e}")
        
    # 2. 匯出為 ONNX 格式
    print("正在將 PyTorch 模型匯出為 ONNX 格式...")
    try:
        sample_data = torch.zeros((1, 3, 224, 224)).cuda()
        torch.onnx.export(
            model,
            sample_data,
            ONNX_MODEL,
            input_names=["input_0"],
            output_names=["output_0"],
            opset_version=11,
            do_constant_folding=True
        )
        print(f"✓ ONNX 匯出成功！已儲存至：{ONNX_MODEL}")
        has_onnx = True
    except Exception as e:
        print(f"❌ 錯誤：ONNX 匯出失敗：{e}")
        has_onnx = False

    # 3. 使用 Nvidia 官方 trtexec 編譯器將 ONNX 編譯為 TensorRT .engine
    if has_onnx and os.path.exists(ONNX_MODEL):
        print("\n正在將 ONNX 編譯為 TensorRT 引擎 (這在 JetBot 上需要 1~2 分鐘，請耐心等候)...\n")
        print("提示：trtexec 會在獨立行程中執行，大幅降低 VRAM 記憶體不足 (OOM) 的機率。")
        
        # 尋找 trtexec 路徑
        trtexec_path = "/usr/src/tensorrt/bin/trtexec"
        if not os.path.exists(trtexec_path):
            trtexec_path = "trtexec"
            
        import subprocess
        
        # 嘗試 FP16 半精度模式編譯
        print("嘗試：FP16 半精度模式編譯...")
        cmd_fp16 = f"{trtexec_path} --onnx={ONNX_MODEL} --saveEngine={ENGINE_MODEL} --fp16 --workspace=1024"
        print(f"執行：{cmd_fp16}")
        
        res = subprocess.run(cmd_fp16, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        
        if res.returncode != 0:
            print("⚠️ 警告：FP16 編譯失敗，正在嘗試自動降級至 FP32 單精度模式編譯...")
            cmd_fp32 = f"{trtexec_path} --onnx={ONNX_MODEL} --saveEngine={ENGINE_MODEL} --workspace=1024"
            print(f"執行：{cmd_fp32}")
            res = subprocess.run(cmd_fp32, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            
        if res.returncode == 0:
            print("✓ TensorRT 引擎編譯成功！")
            
            # 4. 讀取編譯好的引擎位元組，包裝成 torch2trt 專用的 TRTModule state_dict 格式
            print(f"正在將 .engine 包裝為 PyTorch 相容的 {TRT_MODEL}...")
            try:
                with open(ENGINE_MODEL, 'rb') as f:
                    engine_bytes = f.read()
                
                state_dict = {
                    "engine": bytearray(engine_bytes),
                    "input_names": ["input_0"],
                    "output_names": ["output_0"]
                }
                
                os.makedirs(os.path.dirname(TRT_MODEL), exist_ok=True)
                torch.save(state_dict, TRT_MODEL)
                print(f"✓ 轉換與儲存成功！TensorRT 加速模型已就緒。")
            except Exception as e:
                print(f"❌ 錯誤：包裝與儲存 TRTModule 失敗：{e}")
        else:
            print("❌ 錯誤：TensorRT 引擎編譯失敗！詳細資訊：")
            print(res.stderr.decode('utf-8', errors='ignore'))
            print("\n請嘗試重啟 Jupyter Kernel 以釋放更多系統資源。")


---

## 🏁 單元一：純道路跟隨測試 (調校 PID 參數)
此部分只載入 Project 5 的道路循跡模型，適合微調馬達轉向與 PD 參數。

In [ ]:
# A1. 匯入基礎套件
import os
import time
import cv2
import numpy as np
import torch
import torchvision.transforms as transforms
from torch2trt import TRTModule
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg, Robot

# A2. 載入 ResNet-18 TensorRT 模型
device = torch.device('cuda')
model_road = TRTModule()
model_road.load_state_dict(torch.load('road_following_model/best_steering_model_xy_trt.pth'))
print("單元一：道路跟隨模型載入成功 ✓")

In [ ]:
# A3. 初始化相機 (416x416) 與 Robot
camera_r = Camera.instance(width=416, height=416, fps=10)
camera_r.start()
robot_r = Robot()
print("相機 (416x416) 與馬達初始化完成 ✓")

In [ ]:
# A4. 定義影像處理與滑桿介面
mean_r = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std_r = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess_road_only(image):
    image = cv2.resize(image, (224, 224))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.transpose((2, 0, 1))
    image = torch.from_numpy(image).float().to(device).half()
    image = image / 255.0
    image = (image - mean_r[:, None, None]) / std_r[:, None, None]
    return image.unsqueeze(0)

# 建立控制滑桿
speed_slider_r = widgets.FloatSlider(min=0.0, max=0.6, step=0.01, value=0.15, description='Speed')
p_slider_r = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.10, description='P Gain')
d_slider_r = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.02, description='D Gain')
bias_slider_r = widgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=0.00, description='Bias')

# 建立遙測數據顯示
x_text_r = widgets.Label(value="X: 0.000")
y_text_r = widgets.Label(value="Y: 0.000")
steer_text_r = widgets.Label(value="Steering: 0.000")
motor_text_r = widgets.Label(value="L: 0.00, R: 0.00")

# 即時影像 Widget
image_widget_r = widgets.Image(format='jpeg', width=300, height=300)

# 左右排版與顯示
controls_box_r = widgets.VBox([
    widgets.HTML("<b>⚙️ PID 參數調校控制台</b>"),
    speed_slider_r,
    p_slider_r,
    d_slider_r,
    bias_slider_r,
    widgets.HTML("<hr><b>📊 即時遙測數據</b>"),
    x_text_r,
    y_text_r,
    steer_text_r,
    motor_text_r
])

panel_r = widgets.HBox([image_widget_r, controls_box_r])
display(panel_r)

In [ ]:
# A5. 循跡控制迴圈
angle_last_r = 0.0
safety_latched_r = False

def execute_road_only(change):
    global angle_last_r, safety_latched_r
    if safety_latched_r:
        robot_r.stop()
        return
        
    try:
        image = change['new']
        
        # 影像前處理與推論
        road_tensor = preprocess_road_only(image)
        output = model_road(road_tensor).detach().cpu().numpy().flatten()
        x = float(output[0])
        y = float(output[1])
        
        # 計算前向距離與目標角度 (避免除以零或方向反轉)
        y_forward = max(1.0 - y, 0.05)
        angle = np.arctan2(x, y_forward)
        
        # PD 控制器計算轉向
        pid = angle * p_slider_r.value + (angle - angle_last_r) * d_slider_r.value
        angle_last_r = angle
        steering = pid + bias_slider_r.value
        
        # 彎道自動降速 (依轉向量計算速度係數，急彎最低降至設定速度的 60%)
        curve_scale = max(1.0 - abs(steering), 0.6)
        base_speed = speed_slider_r.value * curve_scale
        
        left_motor = max(min(base_speed + steering, 1.0), 0.0)
        right_motor = max(min(base_speed - steering, 1.0), 0.0)
        
        # 驅動馬達
        robot_r.left_motor.value = left_motor
        robot_r.right_motor.value = right_motor
        
        # 更新即時數據顯示
        x_text_r.value = f"X: {x:.3f}"
        y_text_r.value = f"Y: {y:.3f}"
        steer_text_r.value = f"Steering: {steering:.3f} (Scale: {curve_scale:.2f})"
        motor_text_r.value = f"L: {left_motor:.2f}, R: {right_motor:.2f}"
        
        # 繪製綠色路徑點
        display_img = np.copy(image)
        pixel_x = int(x * 208 + 208)
        pixel_y = int(y * 208 + 208)
        cv2.circle(display_img, (pixel_x, pixel_y), 8, (0, 255, 0), -1)
        image_widget_r.value = bgr8_to_jpeg(display_img)
    except Exception as e:
        print(f"❌ [Unit 1 Error] Exception in execute_road_only: {e}")
        robot_r.stop()
        safety_latched_r = True


In [ ]:
# A6. 啟動單元一 (道路跟隨)
angle_last_r = 0.0
safety_latched_r = False
execute_road_only({'new': camera_r.value})
angle_last_r = 0.0
safety_latched_r = False
camera_r.observe(execute_road_only, names='value')
print("🚗 單元一：純道路跟隨自動駕駛啟動！")

In [ ]:
# A7. 安全停止單元一 (切換單元前必執行！)
import gc
try:
    camera_r.unobserve(execute_road_only, names='value')
except Exception as e:
    print(f"Unobserve camera_r failed: {e}")
time.sleep(0.5)
robot_r.stop()
try:
    camera_r.stop()
except Exception as e:
    print(f"Stop camera_r failed: {e}")

# 完全釋放相機 singleton 資源，避免 Resource busy 或解析度鎖定
if hasattr(Camera, '_instance') and Camera._instance is not None:
    Camera._instance = None
gc.collect()
print("單元一已停止，相機與馬達資源已釋放，已重置相機單例 ✓")


---

## 🚥 單元二：純交通路牌辨識測試
此部分只載入 Project 6 的 YOLO 號誌偵測模型，測試自走車看到路牌後的停、走動作是否完全正確。在此模式下，自走車會直線行駛，只反應路牌動作。

In [ ]:
# ========================================================
# 載入 YOLO 推論庫與 C++ 插件 (支援 JetBot 本地與 PC 開發路徑)
# ========================================================
import sys
import os

try:
    import pycuda.autoinit
    from utils.yolo import TRT_YOLO
    print("✓ 成功於本地直接載入 YOLO 推論庫。")
except (ImportError, OSError) as e:
    print(f"直接載入失敗 ({e})，正在嘗試尋找相對路徑之 YOLO 工具庫...")
    original_cwd = os.getcwd()
    yolo_lib_path = os.path.abspath('../road_following/trt_yolov4-tiny-master')
    if not os.path.exists(yolo_lib_path):
        yolo_lib_path = os.path.abspath('../road_following/trt_yolv4-tiny-master')
    
    if os.path.exists(os.path.join(yolo_lib_path, 'utils/yolo.py')):
        os.chdir(yolo_lib_path)
        if yolo_lib_path not in sys.path:
            sys.path.insert(0, yolo_lib_path)
        try:
            import pycuda.autoinit
            from utils.yolo import TRT_YOLO
            print(f"✓ 成功載入 YOLO 插件，工具庫路徑：{yolo_lib_path}")
        finally:
            os.chdir(original_cwd)
    else:
        print(f"❌ 錯誤：找不到 YOLO 工具庫。請確認 utils/yolo.py 與 plugins/ 存在。")
        raise e


In [ ]:
# B2. 載入 YOLOv4-tiny 號誌偵測模型
from utils.yolo import TRT_YOLO
model_sign_only = TRT_YOLO('yolov4-tiny-416', (416, 416), 4)
print("單元二：YOLO 號誌模型載入成功 ✓")

In [ ]:
# B3. 初始化相機 (416x416) 與 Robot
camera_s = Camera.instance(width=416, height=416, fps=10)
camera_s.start()
robot_s = Robot()
print("相機 (416x416) 與馬達初始化完成 ✓")

In [ ]:
# B4. 建立滑桿介面
speed_slider_s = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.18, description='Base Speed')
alert_width_slider_s = widgets.IntSlider(min=20, max=150, step=5, value=50, description='Alert Width')
image_widget_s = widgets.Image(format='jpeg', width=416, height=416)

display(image_widget_s, speed_slider_s, alert_width_slider_s)

In [ ]:
# B5. 路牌偵測控制迴圈
stop_cooldown_s = 0.0
rail_cooldown_s = 0.0
stop_until_s = 0.0  # 新增：非阻塞停車結束時間戳
pedestrian_until_s = 0.0
blocked_latched_s = False
safety_latched_s = False

# 用於追蹤連續偵測幀數的計數器 (0: stop, 1: rail, 2: pedestrian, 3: blocked)
sign_counts_s = {0: 0, 1: 0, 2: 0, 3: 0}

def execute_sign_only(change):
    global stop_cooldown_s, rail_cooldown_s, stop_until_s, pedestrian_until_s, blocked_latched_s, safety_latched_s, sign_counts_s
    if blocked_latched_s or safety_latched_s:
        robot_s.stop()
        return
        
    try:
        img = change['new']
        current_time = time.time()
        
        # 0. 檢查非阻塞停止狀態
        if current_time < stop_until_s:
            robot_s.stop()
            # 標記繪圖 (讓畫面持續更新)
            display_img = np.copy(img)
            cv2.putText(display_img, f"ACTION: Stopping ({stop_until_s - current_time:.1f}s)", (10, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            image_widget_s.value = bgr8_to_jpeg(display_img)
            return
            
        # 偵測路牌
        boxes, confs, clss = model_sign_only.detect(img, conf_th=0.6)
        
        detected_signs = []
        for box, conf, cls in zip(boxes, confs, clss):
            width = box[2] - box[0]
            detected_signs.append({"width": width, "class_id": cls, "box": box})
        # 依寬度排序
        detected_signs.sort(reverse=True, key=lambda x: x["width"])
        
        # 定義各類別個別的框寬門檻 (0: stop=50, 1: rail=30, 2: pedestrian=50, 3: blocked=50)
        slider_val = alert_width_slider_s.value
        width_thresholds = {
            0: slider_val,        # stop
            1: max(20, slider_val - 20), # rail (比 stop 小約 20px, 預設 30px)
            2: slider_val,        # pedestrian
            3: slider_val         # blocked
        }
        
        active_sign = None
        
        # 每幀掃描偵測到的路牌，更新連續計數器
        active_classes_this_frame = set()
        for sign in detected_signs:
            cid = sign["class_id"]
            w = sign["width"]
            
            # 冷卻時間過濾
            if cid == 0 and current_time < stop_cooldown_s: continue
            if cid == 1 and current_time < rail_cooldown_s: continue
            
            # 寬度過濾
            if w >= width_thresholds.get(cid, slider_val):
                active_classes_this_frame.add(cid)
                sign_counts_s[cid] += 1
                
                # 如果連續 2 幀偵測到，且目前沒有更優先的 active_sign
                if sign_counts_s[cid] >= 2 and active_sign is None:
                    active_sign = sign
            
        # 對於本幀未偵測到或未達框寬門檻的類別，重置其連續計數器
        for c in list(sign_counts_s.keys()):
            if c not in active_classes_this_frame:
                sign_counts_s[c] = 0
                
        speed_multiplier = 1.0
        current_action = "Driving Straight"
        
        if active_sign is not None:
            cid = active_sign["class_id"]
            
            if cid == 0:  # stop (停車再開) ➡ 原地停止 2 秒
                current_action = "ACTION: Stop (2s)"
                print("[SIGN] Detected STOP. Stopping for 2s...")
                robot_s.stop()
                stop_until_s = current_time + 2.0
                stop_cooldown_s = current_time + 12.0
                sign_counts_s[0] = 0
                display_img = np.copy(img)
                cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                image_widget_s.value = bgr8_to_jpeg(display_img)
                return
            elif cid == 1:  # rail (鐵路平交道) ➡ 原地停止 5 秒
                current_action = "ACTION: Railway (5s)"
                print("[SIGN] Detected RAILWAY. Stopping for 5s...")
                robot_s.stop()
                stop_until_s = current_time + 5.0
                rail_cooldown_s = current_time + 15.0
                sign_counts_s[1] = 0
                display_img = np.copy(img)
                cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                image_widget_s.value = bgr8_to_jpeg(display_img)
                return
            elif cid == 2:  # pedestrian (當心行人) ➡ 觸發 1 秒減速
                current_action = "ACTION: Pedestrian (Slow 0.7x)"
                pedestrian_until_s = time.time() + 1.0
            elif cid == 3:  # blocked (道路封閉) ➡ 永久停止
                current_action = "ACTION: Blocked! STOP"
                print("[SIGN] Detected BLOCKED. Stopping permanently.")
                robot_s.stop()
                blocked_latched_s = True
                try:
                    camera_s.unobserve(execute_sign_only, names='value')
                except Exception as e:
                    print(f"Unobserve failed: {e}")
                return
                
        # 檢查減速狀態是否在 1 秒維持期內
        if time.time() < pedestrian_until_s:
            speed_multiplier = 0.7
            if active_sign is None:
                current_action = "ACTION: Pedestrian (Latching 0.7x)"
                
        # 驅動馬達直行 (不修正方向)
        base_speed = speed_slider_s.value * speed_multiplier
        robot_s.left_motor.value = base_speed
        robot_s.right_motor.value = base_speed
        
        # 標記繪圖
        display_img = np.copy(img)
        for sign in detected_signs:
            box = sign["box"]
            cid = sign["class_id"]
            cv2.rectangle(display_img, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)
            cv2.putText(display_img, f"ID:{cid} W:{sign['width']} F:{sign_counts_s[cid]}", (box[0], box[1] - 5), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
        image_widget_s.value = bgr8_to_jpeg(display_img)
    except Exception as e:
        print(f"❌ [Unit 2 Error] Exception in execute_sign_only: {e}")
        robot_s.stop()
        safety_latched_s = True


In [ ]:
# B6. 啟動單元二 (路牌辨識)
stop_cooldown_s = 0.0
rail_cooldown_s = 0.0
stop_until_s = 0.0
pedestrian_until_s = 0.0
blocked_latched_s = False
safety_latched_s = False
sign_counts_s = {0: 0, 1: 0, 2: 0, 3: 0}

execute_sign_only({'new': camera_s.value})
stop_cooldown_s = 0.0
rail_cooldown_s = 0.0
stop_until_s = 0.0
pedestrian_until_s = 0.0
blocked_latched_s = False
safety_latched_s = False
sign_counts_s = {0: 0, 1: 0, 2: 0, 3: 0}

camera_s.observe(execute_sign_only, names='value')
print("🚗 單元二：純路牌辨識偵測啟動！")

In [ ]:
# B7. 安全停止單元二 (切換單元前必執行！)
import gc
try:
    camera_s.unobserve(execute_sign_only, names='value')
except Exception as e:
    print(f"Unobserve camera_s failed: {e}")
time.sleep(0.5)
robot_s.stop()
try:
    camera_s.stop()
except Exception as e:
    print(f"Stop camera_s failed: {e}")

# 完全釋放相機 singleton 資源，避免 Resource busy 或解析度鎖定
if hasattr(Camera, '_instance') and Camera._instance is not None:
    Camera._instance = None
gc.collect()
print("單元二已停止，相機與馬達資源已釋放，已重置相機單例 ✓")


---

## 🏁🏁 單元三：合併最終版 (道路跟隨 + 路牌辨識)
雙模型載入並行推論。用大圖 (416x416) 偵測路牌，偵測後即時下採樣 (224x224) 預測方向，配合 PD 運算驅動馬達。此部分為期末驗收使用的最終演示版本。

In [ ]:
# C1. 匯入完整套件
import os
import time
import cv2
import numpy as np
import torch
import torchvision.transforms as transforms
from torch2trt import TRTModule
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg, Robot

device = torch.device('cuda') # 確保定義 device

# 載入 YOLO 推論庫與 C++ 插件 (支援 JetBot 本地與 PC 開發路徑)
import sys
try:
    import pycuda.autoinit
    from utils.yolo import TRT_YOLO
    print("✓ 成功於本地直接載入 YOLO 推論庫。")
except (ImportError, OSError) as e:
    print(f"直接載入失敗 ({e})，正在嘗試尋找相對路徑之 YOLO 工具庫...")
    original_cwd = os.getcwd()
    yolo_lib_path = os.path.abspath('../road_following/trt_yolov4-tiny-master')
    if not os.path.exists(yolo_lib_path):
        yolo_lib_path = os.path.abspath('../road_following/trt_yolv4-tiny-master')
    
    if os.path.exists(os.path.join(yolo_lib_path, 'utils/yolo.py')):
        os.chdir(yolo_lib_path)
        if yolo_lib_path not in sys.path:
            sys.path.insert(0, yolo_lib_path)
        try:
            import pycuda.autoinit
            from utils.yolo import TRT_YOLO
            print(f"✓ 成功載入 YOLO 插件，工具庫路徑：{yolo_lib_path}")
        finally:
            os.chdir(original_cwd)
    else:
        print(f"❌ 錯誤：找不到 YOLO 工具庫。請確認 utils/yolo.py 與 plugins/ 存在。")
        raise e


In [ ]:
# C2. 載入雙模型 (道路跟隨 ResNet TRT + YOLOv4-tiny TRT)
model_road_final = TRTModule()
model_road_final.load_state_dict(torch.load('road_following_model/best_steering_model_xy_trt.pth'))

model_sign_final = TRT_YOLO('yolov4-tiny-416', (416, 416), 4)
print("最終合併版：雙模型載入成功 ✓")


In [ ]:
# C3. 初始化相機 (416x416) 與 Robot
camera_f = Camera.instance(width=416, height=416, fps=10)
camera_f.start()
robot_f = Robot()
print("最終合併版：相機 (416x416) 與馬達初始化完成 ✓")

In [ ]:
# C4. 正規化與滑桿介面
mean_f = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std_f = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess_road_final(image):
    image = cv2.resize(image, (224, 224))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.transpose((2, 0, 1))
    image = torch.from_numpy(image).float().to(device).half()
    image = image / 255.0
    image = (image - mean_f[:, None, None]) / std_f[:, None, None]
    return image.unsqueeze(0)

# 建立控制滑桿
speed_slider_f = widgets.FloatSlider(min=0.0, max=0.6, step=0.01, value=0.15, description='Speed')
p_slider_f = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.10, description='P Gain')
d_slider_f = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.02, description='D Gain')
bias_slider_f = widgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=0.00, description='Bias')
alert_width_slider_f = widgets.IntSlider(min=20, max=150, step=5, value=50, description='Alert Width')

# 建立遙測數據顯示
x_text_f = widgets.Label(value="X: 0.000")
y_text_f = widgets.Label(value="Y: 0.000")
steer_text_f = widgets.Label(value="Steering: 0.000")
motor_text_f = widgets.Label(value="L: 0.00, R: 0.00")
action_text_f = widgets.Label(value="Action: Following Lane")

# 即時影像 Widget
image_widget_f = widgets.Image(format='jpeg', width=416, height=416)

# 左右排版與顯示
controls_box_f = widgets.VBox([
    widgets.HTML("<b>⚙️ 合併版自動駕駛控制台</b>"),
    speed_slider_f,
    p_slider_f,
    d_slider_f,
    bias_slider_f,
    alert_width_slider_f,
    widgets.HTML("<hr><b>📊 即時遙測數據</b>"),
    x_text_f,
    y_text_f,
    steer_text_f,
    motor_text_f,
    action_text_f
])

panel_f = widgets.HBox([image_widget_f, controls_box_f])
display(panel_f)

In [ ]:
# C5. 合併自動駕駛核心迴圈
angle_last_f = 0.0
stop_cooldown_f = 0.0
rail_cooldown_f = 0.0
stop_until_f = 0.0  # 新增：非阻塞停車結束時間戳
pedestrian_until_f = 0.0
blocked_latched_f = False
safety_latched_f = False

# 用於追蹤連續偵測幀數的計數器 (0: stop, 1: rail, 2: pedestrian, 3: blocked)
sign_counts_f = {0: 0, 1: 0, 2: 0, 3: 0}

def execute_final(change):
    global angle_last_f, stop_cooldown_f, rail_cooldown_f, stop_until_f, pedestrian_until_f, blocked_latched_f, safety_latched_f, sign_counts_f
    if blocked_latched_f or safety_latched_f:
        robot_f.stop()
        return
        
    try:
        img = change['new']
        current_time = time.time()
        
        # 0. 檢查非阻塞停止狀態
        if current_time < stop_until_f:
            robot_f.stop()
            # 標記繪圖
            display_img = np.copy(img)
            cv2.putText(display_img, f"ACTION: Stopping ({stop_until_f - current_time:.1f}s)", (10, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            image_widget_f.value = bgr8_to_jpeg(display_img)
            return
            
        # 1. 偵測路牌
        boxes, confs, clss = model_sign_final.detect(img, conf_th=0.6)
        
        detected_signs = []
        for box, conf, cls in zip(boxes, confs, clss):
            width = box[2] - box[0]
            detected_signs.append({"width": width, "class_id": cls, "box": box})
        # 依寬度排序
        detected_signs.sort(reverse=True, key=lambda x: x["width"])
        
        # 定義各類別個別的框寬門檻 (0: stop=50, 1: rail=30, 2: pedestrian=50, 3: blocked=50)
        slider_val = alert_width_slider_f.value
        width_thresholds = {
            0: slider_val,        # stop
            1: max(20, slider_val - 20), # rail (比 stop 小約 20px, 預設 30px)
            2: slider_val,        # pedestrian
            3: slider_val         # blocked
        }
        
        active_sign = None
        
        # 掃描偵測到的路牌，更新連續計數器
        active_classes_this_frame = set()
        for sign in detected_signs:
            cid = sign["class_id"]
            w = sign["width"]
            
            # 冷卻時間過濾
            if cid == 0 and current_time < stop_cooldown_f: continue
            if cid == 1 and current_time < rail_cooldown_f: continue
            
            # 寬度過濾
            if w >= width_thresholds.get(cid, slider_val):
                active_classes_this_frame.add(cid)
                sign_counts_f[cid] += 1
                
                # 如果連續 2 幀偵測到，且目前沒有更優先的 active_sign
                if sign_counts_f[cid] >= 2 and active_sign is None:
                    active_sign = sign
                    
        # 對於本幀未偵測到或未達框寬門檻的類別，重置其連續計數器
        for c in list(sign_counts_f.keys()):
            if c not in active_classes_this_frame:
                sign_counts_f[c] = 0
                
        speed_multiplier = 1.0
        current_action = "Following Lane"
        
        if active_sign is not None:
            cid = active_sign["class_id"]
            
            if cid == 0:  # stop (停車再開) ➡ 原地停止 2 秒
                current_action = "ACTION: Stop (2s)"
                print("[SIGN] Detected STOP. Stopping for 2s...")
                robot_f.stop()
                stop_until_f = current_time + 2.0
                stop_cooldown_f = current_time + 12.0
                sign_counts_f[0] = 0
                display_img = np.copy(img)
                cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                image_widget_f.value = bgr8_to_jpeg(display_img)
                action_text_f.value = f"Action: {current_action}"
                return
                
            elif cid == 1:  # rail (鐵路平交道) ➡ 原地停止 5 秒
                current_action = "ACTION: Railway (5s)"
                print("[SIGN] Detected RAILWAY. Stopping for 5s...")
                robot_f.stop()
                stop_until_f = current_time + 5.0
                rail_cooldown_f = current_time + 15.0
                sign_counts_f[1] = 0
                display_img = np.copy(img)
                cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                image_widget_f.value = bgr8_to_jpeg(display_img)
                action_text_f.value = f"Action: {current_action}"
                return
                
            elif cid == 2:  # pedestrian (當心行人) ➡ 減速 0.7 倍
                current_action = "ACTION: Pedestrian (Slow 0.7x)"
                pedestrian_until_f = time.time() + 1.0
                
            elif cid == 3:  # blocked (道路封閉) ➡ 永久停止
                current_action = "ACTION: Blocked! STOP"
                print("[SIGN] Detected BLOCKED. Stopping permanently...")
                robot_f.stop()
                blocked_latched_f = True
                try:
                    camera_f.unobserve(execute_final, names='value')
                except Exception as e:
                    print(f"Unobserve failed: {e}")
                action_text_f.value = f"Action: {current_action}"
                return
                
        # 檢查減速狀態是否在 1 秒維持期內
        if time.time() < pedestrian_until_f:
            speed_multiplier = 0.7
            if active_sign is None:
                current_action = "ACTION: Pedestrian (Latching 0.7x)"
                
        # 2. 道路循跡推論
        road_tensor = preprocess_road_final(img)
        output = model_road_final(road_tensor).detach().cpu().numpy().flatten()
        x = float(output[0])
        y = float(output[1])
        
        # 計算前向距離與目標角度
        y_forward = max(1.0 - y, 0.05)
        angle = np.arctan2(x, y_forward)
        
        # PD 控制器計算轉向
        pid = angle * p_slider_f.value + (angle - angle_last_f) * d_slider_f.value
        angle_last_f = angle
        steering = pid + bias_slider_f.value
        
        # 彎道自動降速 (依轉向量計算速度係數，急彎最低降至設定速度的 60%)
        curve_scale = max(1.0 - abs(steering), 0.6)
        base_speed = speed_slider_f.value * speed_multiplier * curve_scale
        
        # 計算馬達出力
        left_motor = max(min(base_speed + steering, 1.0), 0.0)
        right_motor = max(min(base_speed - steering, 1.0), 0.0)
        
        # 驅動馬達
        robot_f.left_motor.value = left_motor
        robot_f.right_motor.value = right_motor
        
        # 更新即時數據顯示
        x_text_f.value = f"X: {x:.3f}"
        y_text_f.value = f"Y: {y:.3f}"
        steer_text_f.value = f"Steering: {steering:.3f} (Curve Scale: {curve_scale:.2f})"
        motor_text_f.value = f"L: {left_motor:.2f}, R: {right_motor:.2f}"
        action_text_f.value = f"Action: {current_action}"
        
        # 3. 繪圖標記
        display_img = np.copy(img)
        pixel_x = int(x * 208 + 208)
        pixel_y = int(y * 208 + 208)
        cv2.circle(display_img, (pixel_x, pixel_y), 10, (0, 255, 0), -1)
        
        for sign in detected_signs:
            box = sign["box"]
            cid = sign["class_id"]
            cv2.rectangle(display_img, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)
            cv2.putText(display_img, f"ID:{cid} W:{sign['width']} F:{sign_counts_f[cid]}", (box[0], box[1] - 5), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
                        
        cv2.putText(display_img, current_action, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
        image_widget_f.value = bgr8_to_jpeg(display_img)
    except Exception as e:
        print(f"❌ [Unit 3 Error] Exception in execute_final: {e}")
        robot_f.stop()
        safety_latched_f = True


In [ ]:
# C6. 啟動單元三 (最終合併自動駕駛)
angle_last_f = 0.0
stop_cooldown_f = 0.0
rail_cooldown_f = 0.0
stop_until_f = 0.0
pedestrian_until_f = 0.0
blocked_latched_f = False
safety_latched_f = False
sign_counts_f = {0: 0, 1: 0, 2: 0, 3: 0}

execute_final({'new': camera_f.value})
angle_last_f = 0.0
stop_cooldown_f = 0.0
rail_cooldown_f = 0.0
stop_until_f = 0.0
pedestrian_until_f = 0.0
blocked_latched_f = False
safety_latched_f = False
sign_counts_f = {0: 0, 1: 0, 2: 0, 3: 0}

camera_f.observe(execute_final, names='value')
print("🚗 單元三：雙模型合併自動駕駛啟動！")

In [ ]:
# C7. 安全停止單元三
import gc
try:
    camera_f.unobserve(execute_final, names='value')
except Exception as e:
    print(f"Unobserve camera_f failed: {e}")
time.sleep(0.5)
robot_f.stop()
try:
    camera_f.stop()
except Exception as e:
    print(f"Stop camera_f failed: {e}")

# 完全釋放相機 singleton 資源，避免 Resource busy 或解析度鎖定
if hasattr(Camera, '_instance') and Camera._instance is not None:
    Camera._instance = None
gc.collect()
print("單元三已停止，相機與馬達資源已安全釋放，已重置相機單例 ✓")
